# IPR010047 클러스터 임계값 스윕 — min-seq-id 에 따라 분류가 어떻게 변하는가

2026-04 에 `--min-seq-id 0.70` 으로 219 클러스터를 얻었다. **그 값이 적절한지 판단할 근거가 없다.**
임계값을 0.10~0.90 으로 흔들어 분류가 어디서 안정적이고 어디서 급변하는지 본다.
고정 파라미터 `-c 0.8 --cov-mode 1 -s 7.5` 는 2026-04 와 동일하다. 바꾸면 219 와 비교할 수 없다.

## 전체 흐름

| 절 | 하는 일 | 보고 판단할 것 | mmseqs |
|---|---|---|---|
| 1 설정 | 경로와 상수를 한곳에 모은다 | 고칠 곳은 `REPO`·`BASE` 두 줄뿐 | — |
| 2 환경 | 커널 python, mmseqs 절대경로 | mmseqs 경로가 나오는가 | — |
| 3 입력 확인 | FASTA 와 master 를 센다 | 1,827 레코드 · 219 클러스터 · Q9F8A8 포함 | — |
| **4 시험 실행** | **0.70 하나만 돌린다** | **소요시간·tmp 용량 → 17회분 추정. 219 재현되나** | **1회** |
| 5 전체 스윕 | 나머지 16개 임계값 | 임계값별 클러스터 수 (끊겨도 이어짐) | 16회 |
| 6 지표 | 분할의 모양 | `largest_frac`·`singleton_frac` 이 1에 가까운 구간 = 쓸모없음 | — |
| **7 안정성** | **인접 임계값 ARI** | **안정 구간과 경계. 0.70 이 어디 있는가** ← 이 노트북의 결론 | — |
| 8 ChCODH2 추적 | 참조 클러스터의 변화 | 합병 지점 = 보존도 분석 세트의 경계 | — |
| 9 대조군 | markers.txt 의 동거 | 지금은 accession 1개뿐이라 공집합 | — |
| 10 그림 | 선택 | matplotlib 없으면 건너뜀 | — |
| 11 저장 | tsv 4개 + params.json | 산출 경로 | — |

**4절까지 돌리고 멈춰도 된다.** 거기서 나온 시간·용량 추정을 보고 전체를 돌릴지,
임계값 개수를 줄일지 정하면 된다. 5절은 오래 걸릴 수 있으니 터미널에서 돌리는 쪽이 안전하다.

읽는 방향은 **6·7절이 "임계값을 얼마로 할까"**, **8절이 "그 임계값에서 누구를 분석 대상으로 할까"** 다.

## 작업 폴더

```
/mnt/af2results/mingyu/workspace/ipr010047_mmseqs/
    input/      FASTA, master_1827.tsv   (원본은 다른 곳 — 아래 명령으로 링크)
    script/     이 노트북 (20261025_01_cluster_sweep.ipynb)
    result/     지표·표·params.json, clusters/ 아래에 임계값별 결과
    tmp/        mmseqs 작업 공간 (임계값마다 분리)
```

## 처음 한 번만 — 터미널에서

```bash
BASE=/mnt/af2results/mingyu/workspace/ipr010047_mmseqs
mkdir -p $BASE/{input,script,result,tmp}

ln -s /mnt/af2results/mingyu/chcodh2/protein-matching-IPR010047.fasta $BASE/input/
ln -s /mnt/af2results/mingyu/workspace/docking_simulation/result/homolog_split/master_1827.tsv $BASE/input/

cp ~/office_claude/pipeline/notebooks/cluster_sweep.ipynb \
   $BASE/script/20261025_01_cluster_sweep.ipynb
```

링크 대신 복사(`cp`)해도 된다. 링크가 원본을 단일 출처로 두므로 권장한다.

레포 쪽 파일 이름은 `cluster_sweep.ipynb` 로 고정이고, 워크스테이션으로 가져올 때
`YYYYMMDD_NN_` 을 붙인다. 나중에 갱신본을 받으면 그날 날짜로 새로 복사하면 된다.

## 1. 설정

경로와 상수를 전부 여기 모았다. 다른 셀에는 경로가 흩어져 있지 않으므로,
환경이 바뀌면 이 셀만 고치고 1절부터 다시 실행하면 된다.

`THREADS` 는 mmseqs 가 쓸 코어 수다. RF2-PPI 같은 작업이 같이 돌고 있으면 줄일 것
(인계문서 4절 — `--threads` 를 안 주면 전 코어를 쓴다).

In [ ]:
import json, os, subprocess, sys, time
from pathlib import Path
import pandas as pd

# ── 고칠 곳은 여기뿐이다 ────────────────────────────────────────────
REPO        = Path.home() / "office_claude"                  # 파이프라인 코드 (git)
BASE        = Path("/mnt/af2results/mingyu/workspace/ipr010047_mmseqs")
MMSEQS_PATH = None                           # 2절에서 못 찾을 때만 절대경로를 박는다
THREADS     = min(8, os.cpu_count() or 4)    # mmseqs --threads. 동시 작업이 있으면 줄인다

sys.path.insert(0, str(REPO / "pipeline" / "lib"))
import nb, common
sweep = nb.load_stage("02_cluster_sweep")

INPUT, RESULT, TMP = BASE / "input", BASE / "result", BASE / "tmp"
FASTA   = INPUT / "protein-matching-IPR010047.fasta"
MASTER  = INPUT / "master_1827.tsv"
OUTDIR  = RESULT                             # 표·params.json 이 여기 쌓인다
SWEEP   = RESULT / "clusters"                # 임계값별 _cluster.tsv
for d in (INPUT, RESULT, SWEEP, TMP): d.mkdir(parents=True, exist_ok=True)

REF_ACC    = "Q9F8A8"                        # ChCODH2
STABLE_ARI = 0.90                            # 인접 ARI 가 이 값 이상이면 '안정'
THRESHOLDS = sweep.DEFAULT_THRESHOLDS        # 0.10~0.90, 0.05 간격 = 17개
LABELS     = sweep.make_labels(THRESHOLDS)   # 'id70' 꼴. 파일 이름이 된다

## 2. 커널 환경과 mmseqs 확인

커널이 어느 python 에서 도는지, mmseqs 가 어디 있는지 한 셀에서 본다.

**인계문서 6절의 함정**: Jupyter 커널의 `PATH` 는 로그인 셸의 `PATH` 와 다르다. 터미널에서
`mmseqs` 가 되는데 여기서 `None` 이 나오는 일이 흔하다. 그러면 터미널에서
`readlink -f "$(which mmseqs)"` 를 돌려 절대경로를 확인하고 **1절의 `MMSEQS_PATH` 에 적고
1절부터 다시 실행**하면 된다. 못 찾아도 예외는 안 나니 이 셀이 터지지는 않는다.

In [ ]:
nb.env_report()
print()
MMSEQS = nb.find_mmseqs(MMSEQS_PATH)

## 3. 입력 확인

돌리기 전에 입력이 기대한 것인지 **숫자를 눈으로 보는** 셀이다. FASTA 가 1,827 레코드인지,
기존 master 분할이 219 클러스터인지, 그리고 ChCODH2(Q9F8A8) 가 FASTA 안에 실제로
있는지 확인한다. 숫자가 다르면 아래 전부가 2026-04 결과와 비교 불가이므로 여기서 멈춰야 한다.

In [ ]:
universe, uni = sweep.fasta_universe(FASTA)
print(f"FASTA 레코드      {uni['n_records']}    (기대 1827)")
print(f"고유 accession    {uni['n_unique_accession']}")
print(f"중복/빈 accession {uni['n_duplicate_accession']} / {uni['n_empty_accession']}")
print(f"{REF_ACC} 포함?      {REF_ACC in universe}")

master, minfo = sweep.read_master_partition(MASTER, allow_duplicate_accession=False)
if master is None:
    print(f"\nmaster 없음 ({minfo['reason']}) — 4절의 219 비교는 건너뛴다: {MASTER}")
else:
    uni_idx = pd.Index(universe)
    print(f"\nmaster 분할       {minfo['n_assignments']}개 배정 / "
          f"{minfo['n_clusters']}개 클러스터  (기대 219)")
    print(f"FASTA 와 공통     {len(uni_idx.intersection(master.index))}개  "
          f"(master 에만 {len(master.index.difference(uni_idx))}, "
          f"FASTA 에만 {len(uni_idx.difference(master.index))})")

## 4. 시험 실행 — 0.70 하나만

전체 스윕은 mmseqs 를 **17회** 돌린다. 1회에 얼마나 걸리고 tmp 를 얼마나 쓰는지 아직
아무도 모르므로, 먼저 0.70 하나만 돌려 **소요시간과 tmp 용량을 재고** 덤으로 **219 가
재현되는지** 본다. 여기서 걸린 시간 × 17 이 전체 예상 시간이고, tmp 도 같은 식으로 17배
누적된다(5절은 임계값마다 tmp 디렉터리를 따로 쓴다). 감당 못 할 크기면 임계값을 줄여야 한다.

같이 볼 것: `n_clusters` 가 219 인가, master 와의 `ari` 가 1.0 인가. ARI 가 1.0 이면
분할은 완전히 동일하고 대표 ID 만 바뀐 것이다 (`one_to_one` 수치로 확인). ARI 는 **공통
accession 위에서만** 계산되므로 `공통` 서열 수가 1,827 인지 반드시 같이 본다.

In [ ]:
T0 = 0.70
L0 = sweep.threshold_label(T0)
cmd = sweep.build_cluster_cmd(MMSEQS, FASTA, SWEEP / L0, TMP / L0, T0,
                             cov=sweep.FIXED_COV, cov_mode=sweep.FIXED_COV_MODE,
                             sensitivity=sweep.FIXED_SENSITIVITY, threads=THREADS)
cmds = {L0: " ".join(cmd)}          # 돌린 명령을 전부 모아 11절 params.json 에 남긴다
print(" ".join(cmd), "\n")

t0 = time.time()
subprocess.run(cmd, check=True, stdout=subprocess.DEVNULL)   # 에러는 stderr 로 그대로 보인다
elapsed = time.time() - t0
tmp_mib = sweep.dir_bytes(TMP / L0) / 2**20
print(f"소요 {elapsed:.1f}초   tmp {tmp_mib:.0f} MiB")
print(f"→ {len(THRESHOLDS)}회 예상: 시간 {elapsed * len(THRESHOLDS) / 60:.1f}분, "
      f"tmp 누적 {tmp_mib * len(THRESHOLDS) / 1024:.2f} GiB\n")

df0, _ = sweep.read_cluster_tsv(SWEEP / f"{L0}_cluster.tsv")
p0, _ = sweep.partition_from_cluster_df(df0, L0, allow_duplicate_accession=False)
print(f"n_clusters = {p0.nunique()}   배정 {len(p0)}/{len(universe)}   (2026-04 결과 219)")
if master is not None:
    c = sweep.compare_partitions(p0, master)
    print(f"master 와 ARI {c['ari']:.4f}  NMI {c['nmi']:.4f}  공통 {c['n_common']}서열  "
          f"1:1 대응 {c['one_to_one']}개  분할 {c['n_split']}  합병 {c['n_merged']}")

## 5. 전체 스윕

나머지 임계값을 낮은 쪽부터 돌린다. 임계값마다 한 줄씩 클러스터 수와 소요시간을 찍으므로
기다리는 동안 진행 상황이 보인다. **결과 파일이 이미 있으면 mmseqs 를 건너뛰고 읽기만
하므로**, 중간에 끊겨도 이 셀을 다시 실행하면 이어진다. 0.70 은 4절에서 이미 끝냈다.

건너뛰기 판단은 **파일이 있는지 뿐**이다. 그래서 `-c`·`-s`·입력 FASTA 를 바꿨다면 낡은 결과를
새 파라미터의 결과로 착각한다. 파라미터를 바꿀 때는 `clusters/` 를 비우고 다시 돌릴 것.
배정 서열 수가 FASTA 와 다르면(끊긴 실행이 남긴 잘린 파일) 그 줄에 ⚠ 가 찍힌다.

In [ ]:
partitions, runs = {L0: p0}, {L0: elapsed}

for t, label in zip(THRESHOLDS, LABELS):
    if label in partitions:
        continue
    out_tsv = SWEEP / f"{label}_cluster.tsv"
    c = sweep.build_cluster_cmd(MMSEQS, FASTA, SWEEP / label, TMP / label, t,
                                cov=sweep.FIXED_COV, cov_mode=sweep.FIXED_COV_MODE,
                                sensitivity=sweep.FIXED_SENSITIVITY, threads=THREADS)
    cmds[label] = " ".join(c)
    if not out_tsv.exists():                       # 있으면 건너뛴다 → 재실행 시 이어짐
        t0 = time.time()
        subprocess.run(c, check=True, stdout=subprocess.DEVNULL)
        runs[label] = time.time() - t0
    d, _ = sweep.read_cluster_tsv(out_tsv)
    partitions[label], _ = sweep.partition_from_cluster_df(
        d, label, allow_duplicate_accession=False)
    n = len(partitions[label])
    took = f"{runs[label]:.1f}초" if label in runs else "기존 파일"
    bad = "" if n == len(universe) else f"  ⚠ 배정 {n}/{len(universe)} — 낡았거나 잘린 파일"
    print(f"{label}  seq-id {t:.2f}  {partitions[label].nunique():5d} 클러스터  {took}{bad}")

ORDER = [lb for lb in LABELS if lb in partitions]
print(f"\n완료 {len(ORDER)}/{len(LABELS)},  실행 합계 {sum(runs.values()) / 60:.1f}분")

## 6. 임계값별 지표

임계값마다 분할의 '모양' 을 한 줄로 본다. `n_sequences` 는 배정된 서열 수(1,827 이어야 한다),
`singleton_frac` 은 **클러스터 중** 혼자인 것의 비율(서열 비율이 아니다), `largest_frac` 은
가장 큰 클러스터가 전체 서열의 몇 %를 차지하는지다.
낮은 임계값에서 `largest_frac` 이 1에 가까우면 전부 한 덩어리(해상도 없음), 높은 임계값에서
`singleton_frac` 이 1에 가까우면 아무것도 묶이지 않은 것(역시 쓸모없음)이다.

In [ ]:
metrics = pd.DataFrame([
    dict(label=lb, min_seq_id=sweep.label_to_threshold(lb),
         **common.partition_summary(partitions[lb]))
    for lb in ORDER
]).set_index("label")

metrics[["min_seq_id", "n_sequences", "n_clusters", "n_singletons", "singleton_frac",
         "largest_cluster", "largest_frac", "median_size"]].round(3)

## 7. 인접 임계값 안정성 — 이 노트북의 핵심

인접한 두 임계값의 분할을 ARI 로 비교한다. ARI 가 높게 유지되는 구간은 **임계값을 조금
흔들어도 분류가 같다**는 뜻(안정), 급락하는 지점은 분류가 재편되는 **경계**다. 0.70 이
안정 구간의 안쪽에 있으면 그 값을 계속 쓸 근거가 되고, 경계에 걸쳐 있으면 위험하다.

**⚠ 퇴화 주의.** 전부 한 덩어리인 구간과 전부 싱글턴인 구간도 ARI 는 1.0 이라 '안정' 으로
집계된다. 그것은 안정이 아니라 **"해상도가 쓸모없음"** 이다. 그래서 아래 안정 구간 목록에는
클러스터 수 범위를 같이 찍고, 구간 전체가 퇴화면 ⚠ 를 붙인다. 보통 양쪽 끝이 그렇다.

In [ ]:
adjacent, interp = sweep.adjacent_table(partitions, ORDER, stable_ari=STABLE_ARI,
                                        ref_acc=REF_ACC, max_list=20)
print(f"ARI 중앙값 {interp['ari_median']:.3f}  (최소 {interp['ari_min']:.3f} / "
      f"최대 {interp['ari_max']:.3f})   퇴화 쌍 {interp['n_ari_degenerate']}개")

print(f"\n■ 안정 구간 (인접 ARI >= {STABLE_ARI} 연속) — 클러스터 수를 같이 본다")
for iv in interp["stable_intervals"]:
    seg = adjacent[(adjacent["min_seq_id_low"] >= iv["low"] - 1e-9)
                   & (adjacent["min_seq_id_high"] <= iv["high"] + 1e-9)]
    deg = "  ⚠ 전부 퇴화 = 해상도 없음, 안정이 아니다" if seg["ari_degenerate"].all() else ""
    print(f"   {iv['low']:.2f} ~ {iv['high']:.2f}  {iv['n_steps']}스텝  "
          f"최소 ARI {iv['min_ari']:.3f}  클러스터 "
          f"{seg['n_clusters_coarser'].min()}~{seg['n_clusters_finer'].max()}{deg}")

print("\n■ 경계 (ARI 낮은 쪽부터 5개)")
for b in interp["boundaries"][:5]:
    print(f"   {b['low']:.2f} → {b['high']:.2f}  ARI {b['ari']:.3f}  합병 {b['n_coarser_merged']}개")

hit = interp["reference_in_stable_interval"]
print("\n0.70 : " + (f"{hit['low']:.2f}~{hit['high']:.2f} 안정 구간 안에 있다 "
      f"(구간의 끝점? {interp['reference_in_stable_interval_at_edge']})" if hit
      else "어느 안정 구간에도 속하지 않는다 — 경계에 걸쳐 있다"))
adjacent[["label_low", "label_high", "ari", "nmi", "n_clusters_coarser",
          "n_clusters_finer", "is_stable", "ari_degenerate"]].round(3)

## 8. ChCODH2(Q9F8A8) 클러스터 추적

각 임계값에서 Q9F8A8 이 속한 클러스터의 크기와 대표 ID 를 따라간다. 임계값을 내릴 때
다른 클러스터와 합쳐지는 지점이 보이는데, **보존도 분석 대상 세트를 정하는 근거가 이 표다.**
크기가 너무 작으면 보존도를 계산할 서열이 없고, 갑자기 커지는 지점 아래로는 다른 CODH
계열이 섞여 들어온 것이다. `ref_is_mm_rep` 는 Q9F8A8 자신이 대표로 뽑혔는지 여부다.

In [ ]:
ref_df, ref_per_label, ref_info = sweep.track_reference(
    partitions, ORDER, REF_ACC, max_members=50)
print(f"{REF_ACC} 발견 {ref_info['n_thresholds_found']}/{len(ORDER)} 임계값,  "
      f"클러스터 크기 {ref_info['size_min']} ~ {ref_info['size_max']}")

print("\n■ 임계값을 내릴 때 Q9F8A8 클러스터가 합쳐지는 지점")
for s in interp["ref_merge_steps"]:
    print(f"   {s['high']:.2f} → {s['low']:.2f}   출처 클러스터 "
          f"{s['n_source_clusters']}개, +{s['n_added']}서열")

ref_df[["min_seq_id", "ref_found", "ref_cluster", "ref_cluster_size", "ref_is_mm_rep"]]

## 9. 대조군 추적 (선택)

`pipeline/lib/markers.txt` 에 적은 대조군이 어느 임계값에서 서로 갈라지는지 본다.
**지금 markers.txt 에는 Q9F8A8 하나뿐이라 쌍 비교(`n_pairs`)가 공집합이고, 이 셀은
사실상 8절의 반복이다.** ChCODH1/3/4/5·RrCODH·DvCODH·MtCODH 의 UniProt accession 을
markers.txt 왼쪽 칼럼에 채우면(주석 해제) 그때부터 쌍 비교가 의미를 가진다.

In [ ]:
markers, mk = sweep.read_markers(REPO / "pipeline" / "lib" / "markers.txt")
print(f"markers.txt : {mk['n_markers']}개 — {[m['marker_accession'] for m in markers]}")

mk_df, mk_per_label, mk_info = sweep.track_markers(partitions, ORDER, markers, max_list=20)
print(f"쌍 비교     : {mk_info['n_pairs']}개   (accession 을 2개 이상 채워야 쌍이 생긴다)")
print(f"FASTA 에 없는 마커 : {mk_info['missing_markers']}")
for p in mk_info["pairs"]:
    print(f"   {p['label_a']} vs {p['label_b']}  같은 클러스터인 최고 임계값 "
          f"{p['highest_together']}")

mk_df[mk_df["found"]][["label", "marker_label", "cluster", "cluster_size",
                       "n_co_markers"]].tail(6)

## 10. 그림 (선택)

matplotlib 이 있으면 두 장을 그린다. 왼쪽은 임계값 대 클러스터 수(로그 y축), 오른쪽은
인접 ARI 다. 빨간 점선이 0.70, 회색 점선이 안정 기준선이다. **축 라벨은 영문이다** —
matplotlib 에 한글 폰트가 없는 환경에서는 한글이 □ 로 깨지기 때문이다.
matplotlib 이 없으면 안내만 찍고 넘어간다. 6·7절 표로 판단은 충분히 된다.

In [ ]:
try:
    import matplotlib.pyplot as plt
except ImportError:
    plt = None
    print("matplotlib 이 없다. 6·7절 표로 충분하다. (설치하지 않고 넘어가도 된다)")

if plt is not None:                                  # 축 이름은 한글 글꼴 경고를 피해 영문
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(metrics["min_seq_id"], metrics["n_clusters"], "o-")
    ax[0].set_yscale("log")
    ax[0].set_xlabel("min-seq-id"); ax[0].set_ylabel("n_clusters (log)")
    ax[0].axvline(0.70, ls="--", c="red")
    ax[1].plot(adjacent["min_seq_id_high"], adjacent["ari"], "o-")
    ax[1].axhline(STABLE_ARI, ls="--", c="gray")
    ax[1].axvline(0.70, ls="--", c="red")
    ax[1].set_xlabel("min-seq-id (higher of pair)"); ax[1].set_ylabel("adjacent ARI")
    ax[1].set_ylim(0, 1.05)
    fig.tight_layout()
    plt.show()

## 11. 저장

표 네 개를 tsv 로, 7절 해석을 json 으로, 재현 정보(입력 해시·명령·버전)를
`common.write_params` 로 `params.json` 에 남긴다. 어떤 파일이 어디 생겼는지 출력하므로
다음에 이어 쓸 때 그 경로를 보면 된다. 기존 `master_1827.tsv` 는 읽기만 했고 건드리지 않았다.

In [ ]:
metrics.to_csv(OUTDIR / "sweep_metrics.tsv", sep="\t")
adjacent.to_csv(OUTDIR / "sweep_adjacent.tsv", sep="\t", index=False)
ref_df.to_csv(OUTDIR / "sweep_chcodh2.tsv", sep="\t", index=False)
mk_df.to_csv(OUTDIR / "sweep_markers.tsv", sep="\t", index=False)
(OUTDIR / "interpretation.json").write_text(
    json.dumps(interp, indent=2, ensure_ascii=False, default=str))

common.write_params(
    OUTDIR, "02_cluster_sweep (notebook)",
    inputs=[FASTA, MASTER],
    outputs=sorted(OUTDIR.glob("sweep_*.tsv")),
    commands=[cmds[lb] for lb in ORDER if lb in cmds],
    executed_this_run=sorted(runs),   # 나머지는 이전 실행의 결과 파일을 재사용했다
    mmseqs=MMSEQS, thresholds=THRESHOLDS, stable_ari=STABLE_ARI,
    fixed=dict(c=sweep.FIXED_COV, cov_mode=sweep.FIXED_COV_MODE,
               s=sweep.FIXED_SENSITIVITY, threads=THREADS),
    n_clusters_by_label={lb: int(partitions[lb].nunique()) for lb in ORDER},
)

print(f"저장 위치 {OUTDIR}")
for f in sorted(OUTDIR.glob("*.tsv")) + sorted(OUTDIR.glob("*.json")):
    print(f"   {f.name:24s} {f.stat().st_size:>10,} bytes")